# Similarity matrix

We built one similarity score for every pair of the 445 feature titles, used by all clustering notebooks.

1. Load `data/features_en.json` from `1_preprocesing.ipynb`.
2. TF-IDF similarity of the descriptions.
3. Semantic similarity from Qwen embeddings of title + descriptions.
4. Blend with fuzzy title matching (50% semantic, 25% TF-IDF, 25% fuzzy).
5. Penalise pairs with conflicting concepts and save the matrix to `data/similarity.npz`.

**Input:** `data/features_en.json` from `1_preprocesing.ipynb`, with one record per unique title and its descriptions. The 22 unpaired descriptions are only used for the TF-IDF word weights.

**Dependencies:** see `requirements.txt` (Python 3.13). The first run downloads the Qwen model (~1.2 GB).

In [13]:
import json
import re
from collections import Counter
from pathlib import Path

import numpy as np
from rapidfuzz import fuzz, process, utils
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import TfidfVectorizer

# weights of the three similarity signals
W_SEMANTIC, W_TFIDF, W_FUZZY = 0.50, 0.25, 0.25
W_TITLE = 0.50
PENALTY = 0.30
MODEL_NAME = "Qwen/Qwen3-Embedding-0.6B"
EMBEDDING_PROMPT = "Instruct: Identify product features that describe the same property\nQuery: "

## 1. Load the feature records

Loads the 445 titles and their descriptions from `data/features_en.json`. The 22 unpaired descriptions are kept separately and only used for the TF-IDF word weights.

In [9]:
feature_data = json.loads(Path("data/features_en.json").read_text(encoding="utf8"))
attribute_records = feature_data["attribute_records"]
unpaired_descriptions = feature_data["unpaired_descriptions"]

attributes = [record["title"] for record in attribute_records]
descriptions = [record["descriptions"] for record in attribute_records]

print(len(attributes), "titles,", sum(bool(d) for d in descriptions), "with descriptions,",
      len(unpaired_descriptions), "unpaired descriptions")

445 titles, 433 with descriptions, 22 unpaired descriptions


## 2. TF-IDF similarity of the descriptions

Each title's descriptions are joined into one document. TF-IDF gives rare words more weight than words used everywhere. The word weights are learned from all descriptions but only titles get a row in the matrix. Titles without descriptions get 0 here and rely on the other two signals.

In [ ]:
#here, each title has its document
documents = [" ".join(texts) for texts in descriptions]

# word weights are learned from all descriptions
tfidf = TfidfVectorizer(stop_words="english", sublinear_tf=True)
tfidf.fit([d for d in documents if d] + unpaired_descriptions)

# rows are normalised, so the dot product is the cosine similarity
vectors = tfidf.transform(documents)
tfidf_similarity = (vectors @ vectors.T).toarray()
print("TF-IDF similarity matrix:", tfidf_similarity.shape)

TF-IDF similarity matrix: (445, 445)


## 3. Semantic title-and-description embeddings

The sentence model converts text into numeric vectors whose direction captures contextual meaning. It encodes each unique title and each distinct aligned description once. For an attribute with descriptions, the description vectors are averaged using their occurrence counts, then blended equally with the title vector (`W_TITLE = 0.50`). This gives both the label and its observed context a voice in one vector per attribute. If no aligned description exists, the title vector is used alone. Unpaired descriptions are intentionally excluded here because they have no verified title association. The public model may need to download the first time; no access token or login is required.

In [17]:
import torch

embedding_model = SentenceTransformer(MODEL_NAME, model_kwargs={"dtype": torch.float32})
title_vectors = embedding_model.encode(
    attributes,
    prompt=EMBEDDING_PROMPT,
    normalize_embeddings=True,
    show_progress_bar=True,
)

all_descriptions = sorted({text for texts in descriptions for text in texts})
description_embedding_vectors = embedding_model.encode(
    all_descriptions,
    prompt=EMBEDDING_PROMPT,
    normalize_embeddings=True,
    show_progress_bar=True,
)
description_vector_by_text = {
    text: description_embedding_vectors[index]
    for index, text in enumerate(all_descriptions)
}

attribute_vectors = []
for title_vector, texts in zip(title_vectors, descriptions):
    if texts:
        description_counts = Counter(texts)
        weighted_description_vector = np.average(
            [description_vector_by_text[text] for text in description_counts],
            axis=0,
            weights=list(description_counts.values()),
        )
        vector = W_TITLE * title_vector + (1 - W_TITLE) * weighted_description_vector
        vector = vector / np.linalg.norm(vector)
    else:
        vector = title_vector
    attribute_vectors.append(vector)

attribute_vectors = np.asarray(attribute_vectors)
raw_semantic = attribute_vectors @ attribute_vectors.T
baseline_similarity = np.median(raw_semantic[np.triu_indices(len(attributes), k=1)])
if baseline_similarity >= 1:
    raise ValueError("Semantic similarities cannot be calibrated: median baseline is 1.")

# Calibrate model-specific cosine baselines to 0; identical vectors remain at 1.
semantic = np.clip(
    (raw_semantic - baseline_similarity) / (1 - baseline_similarity),
    0,
    1,
)
print(f"Semantic matrix ready; uncalibrated median baseline: {baseline_similarity:.3f}")

Batches: 100%|██████████| 19/19 [02:07<00:00,  6.72s/it]

Semantic matrix ready; uncalibrated median baseline: 0.566


## 4. Blend the three similarities

Fuzzy matching compares the title wording itself, ignoring word order ("Cycling Gloves" = "Gloves Cycling"). The final similarity is a weighted average: 50% semantic, 25% TF-IDF and 25% fuzzy. We blend the similarity matrices instead of combining the vectors, because embeddings and TF-IDF vectors live in different spaces.

In [ ]:
assert np.isclose(W_SEMANTIC + W_TFIDF + W_FUZZY, 1), "weights must sum to 1"

fuzzy = process.cdist(
    attributes,
    attributes,
    scorer=fuzz.token_sort_ratio,
    processor=utils.default_process,
) / 100

#rescaling fuzzy the same way as semantic
fuzzy_baseline = np.median(fuzzy[np.triu_indices(len(attributes), k=1)])
fuzzy = np.clip((fuzzy - fuzzy_baseline) / (1 - fuzzy_baseline), 0, 1)

similarity = (
    W_SEMANTIC * semantic
    + W_TFIDF *  tfidf_similarity
    + W_FUZZY * fuzzy
)
# A missing description gives no IDF evidence, but every item has zero distance to itself.
np.fill_diagonal(similarity, 1.0)
print(f"Blended similarity matrix: {similarity.shape}")

Blended similarity matrix: (445, 445)
Saved (445, 445) similarity matrix to data\similarity.npz


## 5. Penalty for conflicting concepts

Embeddings see "Deep winter protection" and "Windproof protection" as similar, but they describe different properties (insulation vs. blocking wind). Pairs where one title is only about winter/cold and the other only about wind lose 0.30 similarity (72 pairs). Titles that mention both are not penalised.


> **TODO:** this rule was carried over from an earlier notebook so we need to check it. We need to compare clustering results with and without the penalty once the evaluation exists

In [20]:
CONCEPTS = {
    "winter": r"\b(winter|cold|freez\w*|frost\w*)\b",
    "windproof": r"\b(wind\w*)\b",
}
CONFLICTS = [("winter", "windproof")]

concepts_by_attribute = [
    {
        name
        for name, pattern in CONCEPTS.items()
        if re.search(pattern, title, re.IGNORECASE)
    }
    for title in attributes
]

penalty_matrix = np.zeros_like(similarity)
for first_concept, second_concept in CONFLICTS:
    first_only = np.array(
        [first_concept in concepts and second_concept not in concepts for concepts in concepts_by_attribute]
    )
    second_only = np.array(
        [second_concept in concepts and first_concept not in concepts for concepts in concepts_by_attribute]
    )
    penalty_matrix[np.ix_(first_only, second_only)] = PENALTY
    penalty_matrix[np.ix_(second_only, first_only)] = PENALTY

similarity = np.clip(similarity - penalty_matrix, 0, 1)
print(f"Conflicting attribute pairs penalized: {int(np.count_nonzero(penalty_matrix) // 2)}")

Conflicting attribute pairs penalized: 72


In [ ]:

# Save the shared similarity matrix for all clustering notebooks
output_path = Path("data/similarity.npz")
output_path.parent.mkdir(exist_ok=True)

np.savez_compressed(
    output_path,
    similarity=similarity,                 # 445 x 445, values in [0, 1]
    titles=np.array(attributes),           # row/column order of the matrix
)
print(f"Saved {similarity.shape} similarity matrix to {output_path}")